# Fine-tune LLM using LoRA and QLoRA

In [ ]:
!git clone https://github.com/shosakaue0808/LoRA-QLoRA-studies.git

In [ ]:
!pip install transformers peft

In [ ]:
from transformers import AutoTokenizer, LlamaForCausalLM
from peft import prepare_model_for_kbit_training
from huggingface_hub import login
import torch
from datasets import load_dataset
from dotenv import load_dotenv
import os
# my files
from src/lora_layer import attach_Lora_to_linear




## login Hugging face

In [ ]:
load_dotenv()
token = os.getenv("HF_TOKEN")
login(token)

In [ ]:
# Set device (use GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load dataset

In [ ]:
ds = load_dataset("databricks/databricks-dolly-15k")

# Load Model

In [ ]:
model_id = "meta-llama/Llama-2-7b-hf"

# LoRA base

In [ ]:
base_model_lora = LlamaForCausalLM.from_pretrained(model_id)
base_tokenizer_lora = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf")
base_tokenizer_lora.pad_token = base_tokenizer_lora.eos_token
base_model_lora.config.pad_token_id = base_tokenizer_lora.pad_token_id

# QLoRA base

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
base_model_qlora = LlamaForCausalLM.from_pretrained(model_id, quantization_config=bnb_config)
base_model_qlora.model_name = model_id
base_tokenizer_qlora = AutoTokenizer.from_pretrained(model_id)
base_tokenizer_qlora.pad_token = base_tokenizer_qlora.eos_token
base_model_qlora.config.pad_token_id = base_tokenizer_qlora.pad_token_id

In [ ]:
print(base_model_lora)

In [ ]:
for name, p in base_model_lora.named_parameters():
    if p.requires_grad:
        print(name, p.shape)

In [ ]:
#Set all weights freeze
for param in base_model.parameters():
    param.requires_grad = False

# Attach LoRA layer to all linear layers in base model

attach_Lora_to_linear(base_model_lora)

In [ ]:
# make sure only A, B parameters are trainable
for name, p in base_model_lora.named_parameters():
    if p.requires_grad:
        print(name, p.shape)

In [ ]:
# ratio of trainable / total
total_trainable_params = sum(p.numel() for p in base_model_lora.parameters() if p.requires_grad)
total_prams= sum(p.numel() for p in base_model_lora.parameters())
print(total_trainable_params/total_prams)
